Generative Pre-trained Transformer

In [12]:
import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
data_path = "/home/bukunmi/ml-journey/datasets/shakespearesonnet.txt"

with open(data_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [3]:
corpus = raw_text[:150].strip()
print(f"Sample Corpus Details:\n{"="*30}\n{corpus}\n{"="*30}")

Sample Corpus Details:
I

From fairest creatures we desire increase,
That thereby beauty’s rose might never die,
But as the riper should by time decease,
His tender heir mig


In [4]:
enc = tiktoken.encoding_for_model("gpt-4o")
data_tensor = torch.tensor(enc.encode(corpus), dtype=torch.long)

Batching

In [10]:
block_size = len(data_tensor)

x_batch = data_tensor[:-1].unsqueeze(0)
y_batch = data_tensor[1:].unsqueeze(0)

Embeddings

In [11]:
embedding_dim = 64

Attention

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim, bias=False)
        self.k = nn.Linear(dim, dim, bias=False)
        self.v = nn.Linear(dim, dim, bias=False)
    
    def forward(self, x):
        B, T, C = x.shape
        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        scores = (Q @ K.transpose(-2, -1)) * (C ** -0.05)
        mask = torch.tril(torch.ones(T, T, device=x.device)).view(1, T, T)

        scores = scores.masked_fill(mask == 0, float('-inf'))
        probs = F.softmax(scores, dim=-1)
        return probs @ V

Transformer

In [13]:
class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.attn = CausalAttention(dim)
        self.ln2 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, 4 * dim),
            nn.ReLU(),
            nn.Linear(4 * dim, dim)
        )
    
    def forward(self, x):
        x = self.attn(self.ln1(x))
        x = self.ffn(self.ln2(x))
        return x